In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch11-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["numpy==1.26.4", "rdkit==2026.9.1", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "11-01-why-interpretability.py"

# 群數相同的兩瓶戊醇

本例重算 Joback 與 Reid（1987）發表的常壓沸點方程式，並對照 NIST 所列 2-戊醇和 3-戊醇的資料。 做小練習時，把程式加在下一格末尾，再重跑該格和最後一格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import Draw

# Joback & Reid (1987), Tables II–III. Only neutral acyclic saturated alcohols here.
groups = {
    "CH3": ("[C;!R;X4;H3;+0]", 23.58),
    "CH2": ("[C;!R;X4;H2;+0]", 22.88),
    "CH": ("[C;!R;X4;H1;+0]", 21.74),
    "OH": ("[O;X2;H1;+0]-[C;X4]", 92.88),
}
def group_counts(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or len(Chem.GetMolFrags(mol)) != 1 or mol.GetRingInfo().NumRings():
        raise ValueError("本例只處理單一非環狀飽和醇。")
    covered = []
    counts = {}
    for name, (smarts, _) in groups.items():
        matches = mol.GetSubstructMatches(Chem.MolFromSmarts(smarts))
        counts[name] = len(matches)
        covered.extend(match[0] for match in matches)
    if sorted(covered) != list(range(mol.GetNumAtoms())) or counts["OH"] != 1:
        raise ValueError("有未涵蓋或重複指派的原子，或分子不屬於一元醇。")
    return counts

def boiling_point(counts):
    return 198.2 + sum(n * groups[name][1] for name, n in counts.items())

# NIST Chemistry WebBook, phase change data (CAS 6032-29-7, 584-02-1).
samples = [("2-Pentanol", "CC(O)CCC", 392, 1),
           ("3-Pentanol", "CCC(O)CC", 388, 2)]
fig, axes = plt.subplots(2, 2, figsize=(8.6, 4.7), width_ratios=[1, 1.3],
                         layout="constrained", sharex="col")
for row, (name, smiles, observed, spread) in enumerate(samples):
    counts = group_counts(smiles)
    predicted = boiling_point(counts)
    print(name, counts, f"Joback = {predicted:.2f} K; NIST = {observed} ± {spread} K")
    mol = Chem.MolFromSmiles(smiles)
    oxygen = next(a.GetIdx() for a in mol.GetAtoms() if a.GetSymbol() == "O")
    axes[row, 0].imshow(Draw.MolToImage(mol, size=(440, 150), highlightAtoms=[oxygen]))
    axes[row, 0].set_title(name, loc="left", fontsize=13)
    axes[row, 0].axis("off")
    ax = axes[row, 1]
    ax.errorbar(observed, 0, xerr=spread, fmt="o", ms=9, capsize=5, color="#12736e")
    ax.scatter(predicted, 1, marker="D", s=70, color="#ac542e")
    ax.text(observed, 0.28, f"{observed} ± {spread}", ha="center")
    ax.text(predicted, 1.28, f"{predicted:.2f}", ha="center")
    ax.set(yticks=[0, 1], yticklabels=["NIST", "Joback"], ylim=(-0.48, 1.75), xlim=(382, 411))
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.grid(axis="x", alpha=0.2)
axes[1, 1].set_xlabel("Normal boiling point (K)")
out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
out.mkdir(exist_ok=True)
fig.savefig(out / "11-01-why-interpretability.png", dpi=170)


In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "11-01-why-interpretability.png")))